In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from abc import ABC, abstractmethod
from pathlib import Path

from cashflow import ScorableModelTemplate
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import os

def process_inputs(raw_consumer_file: str, raw_transactions_file: str):
    """Reads, merges, and processes the input data."""
    df_consumer = pd.read_parquet(raw_consumer_file)
    df_transactions = pd.read_parquet(raw_transactions_file)

    # Merge and filter to get only valid data
    merged_df = df_transactions.merge(df_consumer, on="masked_consumer_id")
    filtered_df = merged_df[merged_df["posted_date"]< merged_df["evaluation_date"]]

    # Clean evaluation_date and create evaluation_day for day of week of evaluation_date
    filtered_df['evaluation_date'] = pd.to_datetime(filtered_df['evaluation_date'])
    filtered_df['evaluation_day'] = filtered_df['evaluation_date'].dt.dayofweek

    # Clean posted_date and create posted_day for day of week of posted_date
    filtered_df['posted_date'] = pd.to_datetime(filtered_df['posted_date'])
    filtered_df['posted_day'] = filtered_df['posted_date'].dt.dayofweek

    # Get loan category from masked_consumer_id
    filtered_df['loan_category'] = filtered_df['masked_consumer_id'].str[2].astype(int)
    filtered_df = filtered_df[filtered_df['loan_category'] == 3]
    return filtered_df[['evaluation_date', 'FPF_TARGET', 'total_balance', 'masked_consumer_id']], filtered_df[['masked_consumer_id', 'posted_date', 'amount', 'category', 'masked_transaction_id']]


# === RUNNING THE PROCESS ===

# Path to the input file
input_file1 = "~/private/DSC 190/mlc/test_data/consumer_data.parquet"
input_file2 = "~/private/DSC 190/mlc/test_data/transactions.parquet"

# Create output folder if it doesn't exist
output_dir = "p3_data"
os.makedirs(output_dir, exist_ok=True)

output_1, output2 = process_inputs(input_file1, input_file2)

# Save to same name under the problem_3 folder
output_path = os.path.join(output_dir, os.path.basename(input_file1))
output_1.to_parquet(output_path, index=False)

output_path = os.path.join(output_dir, os.path.basename(input_file2))
output2.to_parquet(output_path, index=False)

print(f"✅ File saved to: {output_path}")


In [ ]:
class ScorableModel(ScorableModelTemplate):

    def predict(self, raw_consumer_file: str, raw_transactions_file: str):
        df_consumers = self.process_inputs(raw_consumer_file, raw_transactions_file)

        def helper(group_df):
            zero_pos = len(group_df[group_df['amount'] > 0]) == 0
            zero_neg = len(group_df[group_df['amount'] < 0]) == 0
            result = {
                'num_transactions': group_df['amount'].count(),
                'total_amount': group_df['amount'].sum(),
                #'fin_balance': group_df['total_balance'].mean(),
                #'total_pos_amount': group_df[group_df['amount'] > 0]['amount'].sum(),
                #'total_neg_amount': group_df[group_df['amount'] < 0]['amount'].sum(),
                #'mean_pos_amount': group_df[group_df['amount'] > 0]['amount'].mean(),
                #'mean_neg_amount': group_df[group_df['amount'] < 0]['amount'].mean(),
                'mean_amount': group_df['amount'].mean(),
                #'amount_mean_median_diff': group_df['amount'].mean() - group_df['amount'].median(),
                #'min_amount': group_df['amount'].min(),
                #'max_amount': group_df['amount'].max(),
                #'common_category': group_df['category'].mode()[0],
                #'date': group_df['evaluation_date'].min().timestamp(),
                #'first_date_posted': group_df['posted_date'].min().timestamp(),
                #'last_date_posted': group_df['posted_date'].max().timestamp(),
                #'range_date_posted': (group_df['posted_date'].max() - group_df['posted_date'].min()).days,
                #'mode_day_posted': group_df['posted_day'].mode()[0],
                #'mean_date_posted': group_df['posted_date'].mean().timestamp(),
                #'date_mean_median_diff_posted': (group_df['posted_date'].mean() - group_df['posted_date'].median()).days,
                'loan_category': group_df['loan_category'].min(),
                'total_pos_amount': group_df[group_df['amount'] > 0]['amount'].sum() if zero_pos else 0,
                'mean_pos_amount':  group_df[group_df['amount'] > 0]['amount'].mean() if zero_pos else 0,
                'total_neg_amount':  group_df[group_df['amount'] < 0]['amount'].sum() if zero_neg else 0,
                'mean_neg_amount': group_df[group_df['amount'] < 0]['amount'].mean() if zero_neg else 0,
                'FPF_TARGET': group_df['FPF_TARGET'].mean()
            }
        
            return pd.Series(result)

        # TRAINING DATA
        df_train = self.process_inputs(
            "~/private/DSC 190/mlc/test_data/consumer_data.parquet",
            "~/private/DSC 190/mlc/test_data/transactions.parquet"
        )
        
        #df_train = df_train[df_train['loan_category'] == 3]
        aggregated_train_data = df_train.groupby('masked_consumer_id').apply(helper).reset_index()
        aggregated_train_encoded = pd.get_dummies(aggregated_train_data.drop(columns=['masked_consumer_id']), drop_first=True)
        
        all_results = pd.Series(0.0, index=df_train.index)

        for category in [1, 2, 3, 4]:
            # Filter training data for this category
            category_encoded = aggregated_train_encoded[aggregated_train_encoded['loan_category'] == category]
            train_target = category_encoded['FPF_TARGET']
            train_features = category_encoded.drop(columns='FPF_TARGET')
        
            # Split for validation
            X_train, X_test, y_train, y_test = train_test_split(train_features, train_target, test_size=0.25, random_state=42)
        
            # Process and filter prediction data
            aggregated_data = df_consumers.groupby('masked_consumer_id').apply(helper).reset_index()
            aggregated_encoded = pd.get_dummies(aggregated_data.drop(columns=['masked_consumer_id']), drop_first=True)
            aggregated_encoded = aggregated_encoded[aggregated_encoded['loan_category'] == category]
        
            target = aggregated_encoded['FPF_TARGET']
            features = aggregated_encoded.drop(columns='FPF_TARGET')
        
            # Train model
            model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', base_score=0.5)
            model.fit(X_train, y_train)
        
            # Predict probabilities
            probs = model.predict_proba(features)[:, 1]
        
            # Assign probs using index alignment
            all_results.loc[aggregated_encoded.index] = probs
        
            # Validation AUC
            print(f"GROUP {category}")
            print("Validate AUC: ", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))
        
        return np.array(all_results.tolist())


    def process_inputs(self, raw_consumer_file: str, raw_transactions_file: str):
        """Input argument will vary. See you competition's template.

        :param raw_files: list of file path strings, depends on competition
        :return: anything needed for you model to make predictions, e.g. features or processed data
        """
        # Read in files
        df_consumer = pd.read_parquet(raw_consumer_file)
        df_transactions = pd.read_parquet(raw_transactions_file)

        # Merge and filter to get only valid data
        merged_df = df_transactions.merge(df_consumer, on = "masked_consumer_id")
        #filtered_df = merged_df[merged_df["posted_date"]< merged_df["evaluation_date"]]
        filtered_df = merged_df
        # Clean evaluation_date and create evaluation_day for day of week of evaluation_date
        filtered_df['evaluation_date'] = pd.to_datetime(filtered_df['evaluation_date'])
        filtered_df['evaluation_day'] = filtered_df['evaluation_date'].dt.dayofweek

        # Clean posted_date and create posted_day for day of week of posted_date
        filtered_df['posted_date'] = pd.to_datetime(filtered_df['posted_date'])
        filtered_df['posted_day'] = filtered_df['posted_date'].dt.dayofweek

        # Get loan category from masked_consumer_id
        filtered_df['loan_category'] = filtered_df['masked_consumer_id'].str[2].astype(int)
        
        return filtered_df


# Intialize, runs: __check_rep__ to validate class
model = ScorableModel() # error will be raised if the above is not implemented correctly

In [4]:
def process_inputs(raw_consumer_file: str, raw_transactions_file: str):
    """Input argument will vary. See you competition's template.

    :param raw_files: list of file path strings, depends on competition
    :return: anything needed for you model to make predictions, e.g. features or processed data
    """
    # Read in files
    df_consumer = pd.read_parquet(raw_consumer_file)
    df_transactions = pd.read_parquet(raw_transactions_file)

    # Merge and filter to get only valid data
    merged_df = df_transactions.merge(df_consumer, on = "masked_consumer_id")
    #filtered_df = merged_df[merged_df["posted_date"]< merged_df["evaluation_date"]]
    filtered_df = merged_df
    # Clean evaluation_date and create evaluation_day for day of week of evaluation_date
    filtered_df['evaluation_date'] = pd.to_datetime(filtered_df['evaluation_date'])
    filtered_df['evaluation_day'] = filtered_df['evaluation_date'].dt.dayofweek

    # Clean posted_date and create posted_day for day of week of posted_date
    filtered_df['posted_date'] = pd.to_datetime(filtered_df['posted_date'])
    filtered_df['posted_day'] = filtered_df['posted_date'].dt.dayofweek

    # Get loan category from masked_consumer_id
    filtered_df['loan_category'] = filtered_df['masked_consumer_id'].str[2].astype(int)
    
    return filtered_df

In [5]:
df_consumers = process_inputs("~/private/DSC 190/mlc/consumers_test_data.parquet", "~/private/DSC 190/mlc/transactions_test_data.parquet")

def helper(group_df):
    zero_pos = len(group_df[group_df['amount'] > 0]) == 0
    zero_neg = len(group_df[group_df['amount'] < 0]) == 0
    result = {
        'num_transactions': group_df['amount'].count(),
        'total_amount': group_df['amount'].sum(),
        #'fin_balance': group_df['total_balance'].mean(),
        #'total_pos_amount': group_df[group_df['amount'] > 0]['amount'].sum(),
        #'total_neg_amount': group_df[group_df['amount'] < 0]['amount'].sum(),
        #'mean_pos_amount': group_df[group_df['amount'] > 0]['amount'].mean(),
        #'mean_neg_amount': group_df[group_df['amount'] < 0]['amount'].mean(),
        'mean_amount': group_df['amount'].mean(),
        #'amount_mean_median_diff': group_df['amount'].mean() - group_df['amount'].median(),
        #'min_amount': group_df['amount'].min(),
        #'max_amount': group_df['amount'].max(),
        #'common_category': group_df['category'].mode()[0],
        #'date': group_df['evaluation_date'].min().timestamp(),
        #'first_date_posted': group_df['posted_date'].min().timestamp(),
        #'last_date_posted': group_df['posted_date'].max().timestamp(),
        #'range_date_posted': (group_df['posted_date'].max() - group_df['posted_date'].min()).days,
        #'mode_day_posted': group_df['posted_day'].mode()[0],
        #'mean_date_posted': group_df['posted_date'].mean().timestamp(),
        #'date_mean_median_diff_posted': (group_df['posted_date'].mean() - group_df['posted_date'].median()).days,
        'loan_category': group_df['loan_category'].min(),
        'total_pos_amount': group_df[group_df['amount'] > 0]['amount'].sum() if zero_pos else 0,
        'mean_pos_amount':  group_df[group_df['amount'] > 0]['amount'].mean() if zero_pos else 0,
        'total_neg_amount':  group_df[group_df['amount'] < 0]['amount'].sum() if zero_neg else 0,
        'mean_neg_amount': group_df[group_df['amount'] < 0]['amount'].mean() if zero_neg else 0,
        'FPF_TARGET': group_df['FPF_TARGET'].mean()
    }

    return pd.Series(result)

# TRAINING DATA
df_train = process_inputs(
    "~/private/DSC 190/mlc/test_data/consumer_data.parquet",
    "~/private/DSC 190/mlc/test_data/transactions.parquet"
)

#df_train = df_train[df_train['loan_category'] == 3]
aggregated_train_data = df_train.groupby('masked_consumer_id').apply(helper).reset_index()
aggregated_train_encoded = pd.get_dummies(aggregated_train_data.drop(columns=['masked_consumer_id']), drop_first=True)

# Process and filter prediction data
aggregated_data = df_consumers.groupby('masked_consumer_id').apply(helper).reset_index()
aggregated_encoded = pd.get_dummies(aggregated_data.drop(columns=['masked_consumer_id']), drop_first=True)

all_results = pd.Series(0.0, index=aggregated_encoded.index)

In [10]:
for category in [1, 2, 3, 4]:
    # Filter training data for this category
    category_encoded = aggregated_train_encoded[aggregated_train_encoded['loan_category'] == category]

    train_target = category_encoded['FPF_TARGET']
    train_features = category_encoded.drop(columns='FPF_TARGET')

    # Split for validation
    X_train, X_test, y_train, y_test = train_test_split(train_features, train_target, test_size=0.25, random_state=42)

    aggregated_cat_encoded = aggregated_encoded[aggregated_encoded['loan_category'] == category]

    target = aggregated_cat_encoded['FPF_TARGET']
    features = aggregated_cat_encoded.drop(columns='FPF_TARGET')

    # Train model
    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', base_score=0.5)
    model.fit(X_train, y_train)

    # Predict probabilities
    probs = model.predict_proba(features)[:, 1]

    # Assign probs using index alignment
    all_results.loc[aggregated_cat_encoded.index] = probs

    # Validation AUC
    print(f"GROUP {category}")
    print("Validate AUC: ", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))


4000
GROUP 1
Validate AUC:  0.5021595528455285
4000
GROUP 2
Validate AUC:  0.540456323442184
4000
GROUP 3
Validate AUC:  0.5469247405895361
4000
GROUP 4
Validate AUC:  0.494290780141844


In [ ]:
for category in [1, 2, 3, 4]:
    # Filter training data for this category
    category_encoded = aggregated_train_encoded[aggregated_train_encoded['loan_category'] == category]

    train_target = category_encoded['FPF_TARGET']
    train_features = category_encoded.drop(columns='FPF_TARGET')

    # Split for validation
    X_train, X_test, y_train, y_test = train_test_split(train_features, train_target, test_size=0.25, random_state=42)

    aggregated_cat_encoded = aggregated_encoded[aggregated_encoded['loan_category'] == category]

    target = aggregated_cat_encoded['FPF_TARGET']
    features = aggregated_cat_encoded.drop(columns='FPF_TARGET')

    # Train model
    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', base_score=0.5)
    model.fit(X_train, y_train)

    # Predict probabilities
    probs = model.predict_proba(features)[:, 1]

    # Assign probs using index alignment
    all_results.loc[aggregated_cat_encoded.index] = probs

    # Validation AUC
    print(f"GROUP {category}")
    print("Validate AUC: ", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))


In [12]:
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import pandas as pd
import numpy as np

# Assuming all_results is already initialized properly
for category in [3]:
    # Filter training data for this category
    category_encoded = aggregated_train_encoded[aggregated_train_encoded['loan_category'] == category]

    train_target = category_encoded['FPF_TARGET']
    train_features = category_encoded.drop(columns='FPF_TARGET')

    # Split for validation
    X_train, X_test, y_train, y_test = train_test_split(train_features, train_target, test_size=0.25, random_state=42)

    # Compute class imbalance weight
    pos = sum(y_train == 1)
    neg = sum(y_train == 0)

    scale_pos_weight = neg / pos if pos > 0 else 1  # Avoid division by zero

    weights = [0.01, 1, 5, 10, 25, 50, 75, 100, 1000]
    param_grid = dict(scale_pos_weight=weights)
    cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', base_score=0.5)
    grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=cv, scoring='roc_auc')
    grid_result = grid.fit(X_train, y_train)
    best_weight = grid_result.best_params_['scale_pos_weight']

    # Train model with computed weight
    model = xgb.XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        base_score=0.04667,
        scale_pos_weight=best_weight
    )
    model.fit(X_train, y_train)

    # Predict probabilities
    aggregated_cat_encoded = aggregated_encoded[aggregated_encoded['loan_category'] == category]
    target = aggregated_cat_encoded['FPF_TARGET']
    features = aggregated_cat_encoded.drop(columns='FPF_TARGET')
    probs = model.predict_proba(features)[:, 1]

    # Assign probs using index alignment
    all_results.loc[aggregated_cat_encoded.index] = probs

    # Validation AUC
    print(f"GROUP {category}")
    print("Validate AUC: ", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))


/home/jkaminsky/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [20:40:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jkaminsky/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [20:40:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jkaminsky/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [20:40:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jkaminsky/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [20:40:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jkaminsky/.local/lib/python3.11/site-packages/

GROUP 3
Validate AUC:  0.5951617214648489


In [ ]:
df_train = process_inputs(
    "~/private/DSC 190/mlc/p3_data/consumer_data.parquet",
    "~/private/DSC 190/mlc/p3_data/transactions.parquet"
)

In [ ]:
df_train.shape